### 1.GET UNIVERSAL CSV:
### ui_df (user_id,item_id,time) sorted by time,
### u_df (reviewerID,user_id) sorted by reviewerID,
### i_df (asin,item_id) sorted by asin


In [ ]:
import random

import pandas as pd
import gzip
import os
import subprocess

# os.chdir("/data/Chester/preprocessing")
os.getcwd()

In [ ]:
def parse(path):
    g = gzip.open(path, 'rb')
    for l in g:
        yield eval(l)

def get_df(path):
    i = 0
    df = {}
    for d in parse(path):
        df[i] = d
        i += 1
    return pd.DataFrame.from_dict(df, orient='index')

In [ ]:
DATASET="Beauty"
RAW_PATH=os.path.join('./',DATASET)
DATA_FILE="reviews_{}_5.json.gz".format(DATASET)
META_FILE="meta_{}.json.gz".format(DATASET)


In [ ]:
# download data if not exists

if not os.path.exists(RAW_PATH):
    subprocess.call('mkdir ' + RAW_PATH, shell=True)
if not os.path.exists(os.path.join(RAW_PATH, DATA_FILE)):
    print('Downloading interaction data into ' + RAW_PATH)
    subprocess.call(
        'cd {} && curl -O http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_{}_5.json.gz'
        .format(RAW_PATH, DATASET), shell=True)
if not os.path.exists(os.path.join(RAW_PATH, META_FILE)):
    print('Downloading item metadata into ' + RAW_PATH)
    subprocess.call(
        'cd {} && curl -O http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/meta_{}.json.gz'
        .format(RAW_PATH, DATASET), shell=True)

In [ ]:
data_df = get_df(os.path.join(RAW_PATH, DATA_FILE))
data_df.head()
ui_df=data_df.rename(columns={'asin':"item_id",'reviewerID':"user_id",'unixReviewTime':'time'})
ui_df=ui_df[['user_id','item_id','time']]

In [ ]:
print(data_df.shape,ui_df.shape)

In [ ]:
ui_df=ui_df.sort_values(by=['time','user_id'],kind='mergesort').reset_index(drop=True)
data_df.head()

In [ ]:
ui_df.head()

### Check nan and duplicates data

In [ ]:
userID,itemID,timestamp="user_id","item_id","time"
ui_df.dropna(subset=[userID,itemID,timestamp],inplace=True)
ui_df.drop_duplicates(subset=[userID,itemID,timestamp],inplace=True)
print(f'After dropped:{ui_df.shape}')
ui_df[:5]

### Check K-core

In [ ]:
from collections import Counter
import numpy as np

min_u_num, min_i_num = 5, 5

def get_illegal_ids_by_inter_num(df, field, max_num=None, min_num=None):
    if field is None:
        return set()
    if max_num is None and min_num is None:
        return set()

    max_num = max_num or np.inf
    min_num = min_num or -1

    ids = df[field].values
    inter_num = Counter(ids)
    ids = {id_ for id_ in inter_num if inter_num[id_] < min_num or inter_num[id_] > max_num}
    print(f'{len(ids)} illegal_ids_by_inter_num, field={field}')

    return ids


def filter_by_k_core(df):
    while True:
        ban_users = get_illegal_ids_by_inter_num(df, field=userID, max_num=None, min_num=min_u_num)
        ban_items = get_illegal_ids_by_inter_num(df, field=itemID, max_num=None, min_num=min_i_num)
        if len(ban_users) == 0 and len(ban_items) == 0:
            return

        dropped_inter = pd.Series(False, index=df.index)
        if userID:
            dropped_inter |= df[userID].isin(ban_users)
        if itemID:
            dropped_inter |= df[itemID].isin(ban_items)
        print(f'{len(dropped_inter)} dropped interactions')
        df.drop(df.index[dropped_inter], inplace=True)
filter_by_k_core(ui_df)
print(f'k-core shape: {ui_df.shape}')
print(f'shape after k-core: {ui_df.shape}')
ui_df[:5]

In [ ]:
i_mapping_file='i_id_mapping.csv'
u_mapping_file='u_id_mapping.csv'

uni_users=sorted(pd.unique(ui_df[userID]))
uni_items=sorted(pd.unique(ui_df[itemID]))
n_users=len(uni_users)
n_items=len(uni_items)
print(f'n_users:{n_users}')
print(f'n_items:{n_items}')
print(f'iteractions:{ui_df.shape[0]}')
print('sparsity percent:{:.2%}'.format(1-ui_df.shape[0]/(n_users*n_items)))

In [ ]:
#start from 0
u_id_map={k:i for i,k in enumerate(uni_users)}
i_id_map={k:i for i,k in enumerate(uni_items)}

ui_df[userID]=ui_df[userID].map(u_id_map)
ui_df[itemID]=ui_df[itemID].map(i_id_map)
ui_df[userID]=ui_df[userID].astype(int)
ui_df[itemID]=ui_df[itemID].astype(int)

u_df = pd.DataFrame(list(u_id_map.items()), columns=['reviewerID', userID])
i_df = pd.DataFrame(list(i_id_map.items()), columns=['asin', itemID])

u_df.to_csv(os.path.join(RAW_PATH, u_mapping_file), sep='\t', index=False)
i_df.to_csv(os.path.join(RAW_PATH, i_mapping_file), sep='\t', index=False)

print('Load mapping')

In [ ]:
ui_df.head()

In [ ]:
u_df.head()

In [ ]:
i_df.head()

In [ ]:
ui_df.to_csv(os.path.join(RAW_PATH, "ui_interaction.csv"), sep='\t', index=False)

### ui_df (user_id,item_id,time) sorted by time,
### u_df (reviewerID,user_id) sorted by reviewerID,
### i_df (asin,item_id) sorted by asin

### #------------------Universal settings completed---------------------


### 2.reindex meta features (only item data)

In [ ]:
import os
import pandas as pd
import gzip


# os.chdir("/data/Chester/preprocessing")
os.getcwd()

DATASET="Beauty"
RAW_PATH=os.path.join('./',DATASET)
META_FILE="meta_{}.json.gz".format(DATASET)
i_id_mapping = 'i_id_mapping.csv'

In [ ]:
#i_df is item mapping csv (asin,item_id)sorted by asin
i_df=pd.read_csv(os.path.join(RAW_PATH,i_id_mapping),sep='\t')
print(f'{i_id_mapping} {i_df.shape}')
i_df.head()

In [ ]:
def parse(path):
    g = gzip.open(path, 'rb')
    for l in g:
        yield eval(l)

def get_df(path):
    i = 0
    df = {}
    for d in parse(path):
        df[i] = d
        i += 1
    return pd.DataFrame.from_dict(df, orient='index')

i_meta_df=get_df(os.path.join(RAW_PATH, META_FILE))

print(f"item_meta_data:{i_meta_df.shape}")
i_meta_df.head()


In [ ]:
#remapping
map_dict=dict(zip(i_df['asin'],i_df["item_id"]))

i_meta_df['item_id']=i_meta_df['asin'].map(map_dict)
i_meta_df.shape

In [ ]:
i_meta_df.dropna(subset=['item_id'],inplace=True)
i_meta_df.shape

In [ ]:
i_meta_df['item_id']=i_meta_df['item_id'].astype(int)
i_meta_df=i_meta_df.sort_values(by=['item_id'],kind='mergesort').reset_index(drop=True)#very important
i_meta_df.head()

In [ ]:
origin_cols=i_meta_df.columns.tolist()

target_cols=[origin_cols[-1]]+origin_cols[:-1]
target_cols

In [ ]:
target_i_df=i_meta_df[target_cols]
target_i_df.to_csv(os.path.join(RAW_PATH,'i_meta_{}.csv'.format(DATASET)),index=False)
target_i_df.head()

In [ ]:
uni_items=target_i_df['item_id'].unique()
print(f'unique items:{len(uni_items)}')
print(f'min/max of unique items:{min(uni_items)},{max(uni_items)}')

### 3.feature encoder

In [ ]:
import os
import numpy as np
import pandas as pd
import random
import torch

seed=123

random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

# os.chdir("/data/Chester/preprocessing")
print(os.getcwd())

In [ ]:
DATASET="Beauty"
RAW_PATH=os.path.join('./',DATASET)
i_meta_df_name='i_meta_{}.csv'.format(DATASET)

item_meta_file=os.path.join(RAW_PATH,i_meta_df_name)
i_meta_df=pd.read_csv(item_meta_file)
i_meta_df.shape

In [ ]:
i_meta_df.head()

In [ ]:
#sentences:title+brand+category+description | ALL have title+description
title_na_df=i_meta_df[i_meta_df['title'].isnull()]
print("title null:",title_na_df.shape)

price_na_df=i_meta_df[i_meta_df['price'].isnull()]
print("price null:",price_na_df.shape)

imUrl_na_df=i_meta_df[i_meta_df['imUrl'].isnull()]
print("imUrl null:",imUrl_na_df.shape)

brand_na_df=i_meta_df[i_meta_df['brand'].isnull()]
print('brand null:',brand_na_df.shape)

categories_na_df=i_meta_df[i_meta_df['categories'].isnull()]
print("categories null:",categories_na_df.shape)

description_na_df=i_meta_df[i_meta_df['description'].isnull()]
print("description null:",description_na_df.shape)


### Text feature processing

In [ ]:
i_meta_df['title']=i_meta_df['title'].fillna(" ")
i_meta_df['brand']=i_meta_df['brand'].fillna(" ")
i_meta_df['description']=i_meta_df['description'].fillna(" ")

In [ ]:
#-------------Text Features------------------
#remove part html:
import re

#---------------re remove html---------------------
pattern = re.compile(r'<.*?>',re.S)

clean_sentences=[]
max_sentences_len=0
min_sentences_len=9999
sum_sentences_len=0
sentences_n=0

for i,row in i_meta_df.iterrows():
    sen=row['title']+" "+row['brand']+" "
    cates=eval(row['categories'])
    if isinstance(cates,list):
        for c in cates[0]:
            sen=sen+c+" "
    sen+=row["description"]
    #----------remove html-------------
    sen = pattern.sub('', sen)
    #----------------------------------
    sen=sen.replace('\n'," ")

    sen_len=len(sen.split(" "))
    max_sentences_len= sen_len if sen_len>max_sentences_len else max_sentences_len
    min_sentences_len=sen_len if sen_len<min_sentences_len else min_sentences_len
    sum_sentences_len+=sen_len

    sentences_n+=1
    clean_sentences.append(sen)

print(len(clean_sentences),sentences_n)
print(f"max_sentences_len:{max_sentences_len}")
print(f"min_sentences_len:{min_sentences_len}")
print(f"avg_sentences_len:{sum_sentences_len/sentences_n}")
#sum sentences avg.len to 77 clip limited,so use bert

In [ ]:
### Hugging face bert
from transformers import BertModel,BertTokenizer
import torch
import os

model_path=os.path.join(os.getcwd(),'bert-base-uncased')
tokenizer = BertTokenizer.from_pretrained(model_path)
model=BertModel.from_pretrained(model_path)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
model.device

In [ ]:
if not os.path.isdir('./text_temp'):
    os.mkdir("./text_temp")

batch=32
debug_sentences=clean_sentences
n=len(debug_sentences)//batch  #cuda OOM

assert len(debug_sentences)>=batch

if not os.path.isdir('./text_temp'):
    os.mkdir("./text_temp")

for i in range(n if len(debug_sentences)%batch==0 else n+1):
    print(f"{i*batch}-{(i+1)*batch}")

    i_batch=debug_sentences[i*batch:(i+1)*batch]
    i_ids_temp = tokenizer(i_batch, max_length=512,truncation=True,padding="max_length",return_tensors='pt')
    i_ids_temp.to(device)
    with torch.no_grad():
        i_bert_output = model(**i_ids_temp)
    i_cls=i_bert_output.last_hidden_state[:,0,:].to("cpu")
    torch.save(i_cls,f"./text_temp/{i}.pt")
    print(i_cls.shape)

    del i_ids_temp,i_bert_output,i_cls
    torch.cuda.empty_cache()

test_all=[]
for i in range(n if len(debug_sentences)%batch==0 else n+1):
    x=torch.load(f"./text_temp/{i}.pt")
    test_all.append(x)

text_features=torch.cat(test_all,dim=0)
torch.save(text_features,os.path.join(RAW_PATH,"text_feat.pt"))

import shutil
shutil.rmtree("./text_temp")

In [ ]:
text_features.shape

In [ ]:
#sample random text to contrast lable
model.to("cpu")
nnn=64
random_i_feature=tokenizer(clean_sentences[-10:],max_length=512,truncation=True,padding="max_length",return_tensors='pt')
random_i_feature=model(**random_i_feature)
random_i_feature=random_i_feature.last_hidden_state[:,0,:]

print(torch.allclose(random_i_feature,text_features[-10:],1e-1))
print(random_i_feature==text_features[-10:])

# torch.allclose(text_features,zz,atol=1e-5)

In [ ]:
model.to("cpu")
del model
torch.cuda.empty_cache()

### Image Feature processing

In [ ]:
#-------------Image Features------------------
imUrl_na_df=i_meta_df[i_meta_df['imUrl'].isnull()]
print(f"image NaN count:{len(imUrl_na_df[['item_id','imUrl']])}")
print(imUrl_na_df[['item_id',"imUrl"]])

In [ ]:
list_n=0
string_n=0
for i,row in i_meta_df.iterrows():
    if isinstance(row['imUrl'],list):
        list_n+=1
    if isinstance(row['imUrl'],str):
        string_n+=1
list_n
#Some origin item has many image url like:[....jpg,....jpg,....jpg]

In [ ]:
len(i_meta_df['imUrl'])

In [ ]:
i_meta_df['imUrl']

In [ ]:
import re

# pattern = re.compile('(\._).*(_\.)',re.S)
# suffix=pattern.findall(row['imUrl'])


img_list=[]
count_NaN=0
count_suffix=0

resolution_suffix= {'._SX300_.':0,'._SY300_.':0,'no_suffix':0}
for i,row in i_meta_df.iterrows():
    # print(type(row['imUrl']))
    if isinstance(row['imUrl'],float):
        count_NaN+=1
        continue
    if isinstance(row['imUrl'],str):

        if '._SX300_.' in row['imUrl']:
            resolution_suffix['._SX300_.']+=1
        elif '._SY300_.' in row['imUrl']:
            resolution_suffix['._SY300_.']+=1
        else:
            print(i,row['imUrl'])
            resolution_suffix['no_suffix']+=1
        count_suffix+=1

print(f"NaN:{count_NaN},{resolution_suffix}:{count_suffix},suffix_sum:{sum([resolution_suffix[i] for i in resolution_suffix])},item_n:{len(i_meta_df)}")

In [ ]:
#nan unvalid ---------------------=======features Mean


from PIL import Image
import requests
from transformers import AutoProcessor, CLIPModel
import  time

model = CLIPModel.from_pretrained("./clip-vit-base-patch32",use_safetensors=True)
processor = AutoProcessor.from_pretrained("./clip-vit-base-patch32")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)

DATASET="Beauty"
RAW_PATH=os.path.join('./',DATASET)
Image_PATH=os.path.join(RAW_PATH,'images')

img_nan_id=[]
img_bad_id=[]
bad_img_count=0

img_features=[]

if not os.path.isdir(Image_PATH):
    os.mkdir(Image_PATH)

start=time.time()

for i,row in i_meta_df.iterrows():
    if i%50==0:
        if i==0:
            t1=time.time()
        else:
            print("iteration:",i,"\t\ttime:{:.2f}s".format(time.time()-t1))
            t1=time.time()
    if row['imUrl'] is np.nan:
        print(f"{row['item_id']} is nan")
        img_nan_id.append(row['item_id'])
        img_features.append(torch.zeros(1,512).to(device))
    else:
        try:
            image=Image.open(requests.get(row['imUrl'],stream=True).raw)
            image.save(os.path.join(Image_PATH,str(row['item_id']))+".jpg")
            inputs = processor(images=image, return_tensors="pt")
            inputs.to(device)
            with torch.no_grad():
                img_ft = model.get_image_features(**inputs)
            img_features.append(img_ft)
        except:
            print(f'{row["item_id"]}:{row["imUrl"]} is a invalid url!')
            img_bad_id.append(row['item_id'])
            bad_img_count+=1
            img_features.append(torch.zeros(1,512).to(device))

print("abstract time ends.Total time:{:.2f}s".format(time.time()-start))

pt_img_features=torch.stack(img_features)
pt_img_features=pt_img_features.squeeze()
pt_img_mean=torch.mean(pt_img_features,dim=0)
print(pt_img_features.shape)

for i in img_nan_id:
    pt_img_features[i]=pt_img_mean

for i in img_bad_id:
    pt_img_features[i]=pt_img_mean


print(f"invalid image url count:{bad_img_count}")

torch.save(pt_img_features.to("cpu"),os.path.join(RAW_PATH,"img_feat.pt"))
print("success!")

### Test feature match!

In [ ]:
def test_image(url,n):
    image=Image.open(requests.get(url,stream=True).raw)

    inputs = processor(images=image, return_tensors="pt")
    inputs.to(device)
    with torch.no_grad():
        img_ft = model.get_image_features(**inputs)
    print(pt_img_features[n].to("cpu")==img_ft.to("cpu"))

In [ ]:
test_image("http://ecx.images-amazon.com/images/I/21iMxsyDBRL._SX300_.jpg",0)

In [ ]:
i_meta_df['imUrl'][0]

In [ ]:
text_features=torch.load(os.path.join(RAW_PATH,"text_feat.pt"))
text_features.shape

In [ ]:
len(clean_sentences)

In [ ]:
# 加载bert模型

from transformers import BertModel,BertTokenizer
import torch
import os

model_path=os.path.join(os.getcwd(),'bert-base-uncased')
tokenizer = BertTokenizer.from_pretrained(model_path)
model=BertModel.from_pretrained(model_path)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
random_i_feature=tokenizer(clean_sentences[-10:],max_length=512,truncation=True,padding="max_length",return_tensors='pt').to(device)
random_i_feature=model(**random_i_feature)
random_i_feature=random_i_feature.last_hidden_state[:,0,:]

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
text_features=text_features.to(device)

def test_text(a,b=None):
    if b is not None:
        temp=tokenizer(clean_sentences[a:b],max_length=512,truncation=True,padding="max_length",return_tensors='pt').to(device)
        temp=model(**temp)
        temp=temp.last_hidden_state[:,0,:]
        # print(text_features.device,'++',temp.device)
        print(torch.allclose(temp,text_features[a:b],1e-1))
        print(temp==text_features[a:b])

        print(temp)
        print(text_features[a:b])
    else:
        temp=tokenizer(clean_sentences[a],max_length=512,truncation=True,padding="max_length",return_tensors='pt')
        temp=model(**temp)
        temp=temp.last_hidden_state[:,0,:]
        print(torch.allclose(temp,text_features[a],1e-1))
        print(temp==text_features[a])

        print(temp)
        print(text_features[a])

In [ ]:
test_text(-5,-1)

In [ ]:
model.eval()